 **1: Conceptual Answers**
***1. Difference between "Love" and "love" in NLP***
"Love" and "love" are treated as different tokens in case-sensitive systems.
This increases vocabulary size unnecessarily.
Hence, we convert text to lowercase to ensure consistency.
***2. What happens if stopwords are not removed?***
Model may focus on irrelevant frequent words (like "is", "the").
Increases noise and reduces model performance.
Slows down processing due to larger token set.
***3. When removing stopwords is harmful?***
Sentiment Analysis
Example: "I am not happy" → removing "not" changes meaning.
Question Answering Systems
Example: "What is AI?" → removing "what" loses intent.
***4. Stemming vs Lemmatization***
Feature	Stemming	Lemmatization
Approach	Rule-based	Vocabulary-based
Output	Root form (may be incorrect)	Meaningful base word
Example	"running" → "run"	"better" → "good"
Accuracy	Low	High

2: Preprocessing Function

In [1]:
import re

def preprocess_text(text):
    if not text or not isinstance(text, str):
        return [], ""

    # 1. Lowercase
    text = text.lower()

    # 2. Remove URLs & emails
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)

    # 3. Remove numbers
    text = re.sub(r'\d+', '', text)

    # 4. Handle repeated characters (soooo → soo)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 5. Remove special characters & emojis
    text = re.sub(r'[^a-z\s]', '', text)

    # 6. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # 7. Tokenization
    tokens = text.split()

    # 8. Remove short tokens (≤2) except "no", "not"
    tokens = [word for word in tokens if len(word) > 2 or word in ['no', 'not']]

    # 9. Clean sentence
    cleaned_sentence = " ".join(tokens)

    return tokens, cleaned_sentence

3: Stress Testing

In [2]:
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product ",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

results = []

for sentence in test_sentences:
    tokens, clean = preprocess_text(sentence)
    results.append((sentence, tokens, clean))

for r in results:
    print("\nOriginal:", r[0])
    print("Tokens:", r[1])
    print("Cleaned:", r[2])


Original: Get 100% FREE access now!!!
Tokens: ['get', 'free', 'access', 'now']
Cleaned: get free access now

Original: I absolutely looooved this product 
Tokens: ['absolutely', 'looved', 'this', 'product']
Cleaned: absolutely looved this product

Original: Worst service ever... 0/10
Tokens: ['worst', 'service', 'ever']
Cleaned: worst service ever

Original: Call me at 9876543210
Tokens: ['call']
Cleaned: call

Original: This is THE best course!!!
Tokens: ['this', 'the', 'best', 'course']
Cleaned: this the best course

Original: Visit https://openai.com now!
Tokens: ['visit', 'now']
Cleaned: visit now

Original: Nooooo this is baaad!!!
Tokens: ['noo', 'this', 'baad']
Cleaned: noo this baad

Original: OK OK OK I got it
Tokens: ['got']
Cleaned: got

Original: Win $$$ now!!! Limited offer!!!
Tokens: ['win', 'now', 'limited', 'offer']
Cleaned: win now limited offer

Original: I am not happy with this
Tokens: ['not', 'happy', 'with', 'this']
Cleaned: not happy with this


4: Token Analytics

In [3]:
import numpy as np

analytics = []

for sentence, tokens, clean in results:
    if len(tokens) == 0:
        avg_len = 0
    else:
        avg_len = np.mean([len(t) for t in tokens])

    analytics.append({
        "sentence": sentence,
        "total_tokens": len(tokens),
        "unique_tokens": len(set(tokens)),
        "avg_token_length": avg_len
    })

for a in analytics:
    print(a)

{'sentence': 'Get 100% FREE access now!!!', 'total_tokens': 4, 'unique_tokens': 4, 'avg_token_length': np.float64(4.0)}
{'sentence': 'I absolutely looooved this product ', 'total_tokens': 4, 'unique_tokens': 4, 'avg_token_length': np.float64(6.75)}
{'sentence': 'Worst service ever... 0/10', 'total_tokens': 3, 'unique_tokens': 3, 'avg_token_length': np.float64(5.333333333333333)}
{'sentence': 'Call me at 9876543210', 'total_tokens': 1, 'unique_tokens': 1, 'avg_token_length': np.float64(4.0)}
{'sentence': 'This is THE best course!!!', 'total_tokens': 4, 'unique_tokens': 4, 'avg_token_length': np.float64(4.25)}
{'sentence': 'Visit https://openai.com now!', 'total_tokens': 2, 'unique_tokens': 2, 'avg_token_length': np.float64(4.0)}
{'sentence': 'Nooooo this is baaad!!!', 'total_tokens': 3, 'unique_tokens': 3, 'avg_token_length': np.float64(3.6666666666666665)}
{'sentence': 'OK OK OK I got it', 'total_tokens': 1, 'unique_tokens': 1, 'avg_token_length': np.float64(3.0)}
{'sentence': 'Win $$$

5: Frequency Analysis

In [4]:
from collections import Counter

all_tokens = []

for _, tokens, _ in results:
    all_tokens.extend(tokens)

counter = Counter(all_tokens)

print("Top 10 frequent:", counter.most_common(10))
print("Least 5 frequent:", counter.most_common()[-5:])

Top 10 frequent: [('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('looved', 1), ('product', 1), ('worst', 1), ('service', 1)]
Least 5 frequent: [('limited', 1), ('offer', 1), ('not', 1), ('happy', 1), ('with', 1)]


6: Full Pipeline

In [5]:
def full_pipeline(text_list):
    all_tokens = []
    clean_sentences = []

    for text in text_list:
        tokens, clean = preprocess_text(text)
        all_tokens.extend(tokens)
        clean_sentences.append(clean)

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }

7: Error Handling Testing

In [6]:
edge_cases = ["", "😂😂😂😂", "123456789"]

for case in edge_cases:
    tokens, clean = preprocess_text(case)
    print("\nInput:", case)
    print("Tokens:", tokens)
    print("Cleaned:", clean)


Input: 
Tokens: []
Cleaned: 

Input: 😂😂😂😂
Tokens: []
Cleaned: 

Input: 123456789
Tokens: []
Cleaned: 
